# exp_002 — Quantization with llama.cpp GGUF

Load measured summaries and a resolved artifact manifest. The notebook does not run the benchmark or invent missing measurements.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('../..', 'src').resolve()))
from llm_lab.analysis.quantization import recommend_baseline, tradeoff_rows
from llm_lab.quantization import QuantizationManifest

RESULTS_DIR = Path('results')
SUMMARY_PATH = Path('results/processed/summary.csv')
MANIFEST_PATH = Path('results/manifest.json')
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f'measured summary is required: {SUMMARY_PATH}')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'resolved manifest is required: {MANIFEST_PATH}')

manifest = QuantizationManifest.from_record(json.loads(MANIFEST_PATH.read_text()))
summaries = pd.read_csv(SUMMARY_PATH)
required_columns = {
    'condition_id', 'scored_n', 'accuracy', 'median_ttft_s',
    'median_prefill_tokens_per_second', 'median_decode_tokens_per_second',
    'median_peak_memory_bytes',
}
missing_columns = required_columns - set(summaries.columns)
if missing_columns:
    raise ValueError(f'summary is missing required columns: {sorted(missing_columns)}')

rows = tradeoff_rows(summaries.to_dict('records'), manifest)
frame = pd.DataFrame(rows)
frame

In [ ]:
def accuracy_vs_memory(frame: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(frame['median_peak_memory_bytes'] / 1e9, frame['accuracy'])
    for _, row in frame.iterrows():
        ax.annotate(row['label'], (row['median_peak_memory_bytes'] / 1e9, row['accuracy']))
    ax.set_xlabel('Median peak memory (GB)')
    ax.set_ylabel('Task accuracy')
    ax.set_title('Accuracy vs peak memory')
    ax.set_ylim(0, 1.05)
    return fig

RESULTS_DIR.joinpath('figures').mkdir(parents=True, exist_ok=True)
accuracy_figure = accuracy_vs_memory(frame)
accuracy_figure.savefig(RESULTS_DIR / 'figures' / 'accuracy-vs-memory.png', dpi=160, bbox_inches='tight')
accuracy_figure

In [ ]:
def speed_vs_memory(frame: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
    x = frame['median_peak_memory_bytes'] / 1e9
    axes[0].scatter(x, frame['median_prefill_tokens_per_second'])
    axes[1].scatter(x, frame['median_decode_tokens_per_second'])
    for axis, column in zip(axes, ('median_prefill_tokens_per_second', 'median_decode_tokens_per_second')):
        for _, row in frame.iterrows():
            axis.annotate(row['label'], (row['median_peak_memory_bytes'] / 1e9, row[column]))
        axis.set_xlabel('Median peak memory (GB)')
        axis.set_ylabel('tokens / second')
    axes[0].set_title('Prefill speed vs memory')
    axes[1].set_title('Decode speed vs memory')
    return fig

speed_figure = speed_vs_memory(frame)
speed_figure.savefig(RESULTS_DIR / 'figures' / 'speed-vs-memory.png', dpi=160, bbox_inches='tight')
speed_figure

In [ ]:
recommendation = recommend_baseline(rows, accuracy_tolerance=0.02)
print({
    'recommended_condition': recommendation['condition_id'],
    'recommended_label': recommendation['label'],
    'accuracy': recommendation['accuracy'],
    'artifact_size_bytes': recommendation['artifact_size_bytes'],
    'best_accuracy': recommendation['best_accuracy'],
    'accuracy_tolerance': recommendation['accuracy_tolerance'],
})